[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/atoms_nuclei.ipynb)

# Atoms and nuclei full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: alpha particles flying past a nucleus, forty thousand of
them counted by angle beside Rutherford's formula, the "pudding" atom that could not turn them, Bohr's orbits, energy
levels and the lines of hydrogen, the binding-energy curve built from measured atomic masses, the energy of any nuclear
reaction, and the Sun's hydrogen. You can move all of them yourself: the alpha's **energy**, the **target**, the
**number** of alphas, the **jump** an electron makes, the **nucleus** on the curve, the **reaction**, and a star's
**power**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more (a question that wants more decimals says so). Big and
small numbers in E notation: `1.66e-27` for 1.66 × 10⁻²⁷, `9e16` for 9 × 10¹⁶ (`9*10^16` works too). You can type the
arithmetic: `2*29*1.44/6`, `1240/500`, `1.2*40**(1/3)`, `0.05*931.5`. Numbers without thousands separators:
`1150.25`, not `1,150.25`. For several answers, separate them with commas, in the order the question lists them (the
box's label shows the order). You can also put your answer between the quotes in a cell's last line,
`practice("P1.3", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula, direction and constant from the course on two pages:
[atoms and nuclei cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/atoms_nuclei_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the mass table, the formulas and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # paths and curves in chalk, labels dim, ghost lines
MINT, CORAL, VIOLET, GOLD = "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# alphas, nuclei and curves CHALK · force CORAL · energy VIOLET · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (CODATA 2022; the course's handy products) ----
E_CH = 1.602176634e-19     # C, the charge e; also the joules in one electron volt
H = 6.62607015e-34         # J s, Planck's constant
C = 299792458.0            # m/s, the speed of light
EPS0 = 8.8541878188e-12    # F/m
M_E = 9.1093837139e-31     # kg, the electron
K = 1 / (4 * math.pi * EPS0)   # N m²/C², "k": 8.99 × 10⁹
U_KG = 1.66053906892e-27   # kg, one atomic mass unit
U_MEV = 931.49410372       # MeV, one atomic mass unit as energy: 931.5
N_A = 6.02214076e23        # atoms in a mole
EV, MEV = E_CH, 1e6 * E_CH
HC = H * C / E_CH * 1e9    # eV nm, "twelve forty": 1239.84
KE2 = K * E_CH ** 2 / MEV * 1e15   # MeV fm (the same number in eV nm): 1.44
R0 = 1.2                   # fm, the nuclear radius constant
L_SUN = 3.828e26           # W, the Sun's power (the nominal solar luminosity)
M_SUN = 1.9884e30          # kg
YEAR = 365.25 * 86400.0    # s

# ---- the mass table: (element, A): (Z, atomic mass in u, binding energy per nucleon in MeV as the evaluation lists it)
# AME2020, the 2020 atomic mass evaluation (Chinese Physics C 45, 030003, 2021)
MASS = {
    ("n", 1): (0, 1.008664916, 0.0000), ("H", 1): (1, 1.007825032, 0.0000), ("H", 2): (1, 2.014101778, 1.1123),
    ("H", 3): (1, 3.016049281, 2.8273), ("He", 3): (2, 3.016029322, 2.5727), ("He", 4): (2, 4.002603254, 7.0739),
    ("Li", 6): (3, 6.015122887, 5.3323), ("Li", 7): (3, 7.016003434, 5.6064), ("Be", 9): (4, 9.012183062, 6.4627),
    ("B", 11): (5, 11.009305166, 6.9277), ("C", 12): (6, 12., 7.6801), ("C", 14): (6, 14.003241989, 7.5203),
    ("N", 14): (7, 14.003074004, 7.4756), ("O", 16): (8, 15.994914619, 7.9762), ("F", 19): (9, 18.998403162, 7.7790),
    ("Ne", 20): (10, 19.992440175, 8.0322), ("Na", 23): (11, 22.989769282, 8.1115),
    ("Mg", 24): (12, 23.985041689, 8.2607), ("Al", 27): (13, 26.981538408, 8.3316),
    ("Si", 28): (14, 27.976926534, 8.4477), ("P", 31): (15, 30.973761998, 8.4812),
    ("S", 32): (16, 31.972071174, 8.4931), ("Ar", 40): (18, 39.962383122, 8.5953),
    ("Ca", 40): (20, 39.96259085, 8.5513), ("Ti", 48): (22, 47.947940677, 8.7230),
    ("Cr", 52): (24, 51.940504714, 8.7760), ("Fe", 56): (26, 55.934935537, 8.7904),
    ("Ni", 58): (28, 57.93534165, 8.7321), ("Ni", 62): (28, 61.928344753, 8.7946),
    ("Cu", 63): (29, 62.929597119, 8.7521), ("Cu", 65): (29, 64.927789476, 8.7571),
    ("Zn", 64): (30, 63.929141776, 8.7359), ("Ge", 74): (32, 73.92117776, 8.7252),
    ("Kr", 84): (36, 83.911497727, 8.7174), ("Kr", 92): (36, 91.926173092, 8.5127),
    ("Sr", 88): (38, 87.905612253, 8.7326), ("Sr", 94): (38, 93.915355641, 8.5938),
    ("Zr", 90): (40, 89.904698755, 8.7100), ("Mo", 98): (42, 97.905403609, 8.6352),
    ("Pd", 106): (46, 105.903480287, 8.5800), ("Cd", 114): (48, 113.903364998, 8.5315),
    ("Sn", 120): (50, 119.902202557, 8.5045), ("Te", 130): (52, 129.906222745, 8.4303),
    ("Xe", 132): (54, 131.904155083, 8.4276), ("Xe", 140): (54, 139.921645814, 8.2909),
    ("Ba", 138): (56, 137.905247059, 8.3934), ("Ba", 141): (56, 140.914403653, 8.3261),
    ("Nd", 144): (60, 143.910092798, 8.3269), ("Sm", 152): (62, 151.919738646, 8.2441),
    ("Gd", 158): (64, 157.9241112, 8.2018), ("Dy", 164): (66, 163.929180819, 8.1587),
    ("Yb", 174): (70, 173.938867545, 8.0838), ("Hf", 180): (72, 179.946559537, 8.0349),
    ("W", 184): (74, 183.95093318, 8.0051), ("Pt", 195): (78, 194.964794325, 7.9266),
    ("Au", 197): (79, 196.966570103, 7.9157), ("Hg", 202): (80, 201.970643604, 7.8969),
    ("Pb", 206): (82, 205.97446521, 7.8754), ("Pb", 208): (82, 207.976652005, 7.8675),
    ("Bi", 209): (83, 208.980398599, 7.8480), ("Po", 210): (84, 209.982873686, 7.8343),
    ("Rn", 222): (86, 222.017576017, 7.6945), ("Ra", 226): (88, 226.025408186, 7.6620),
    ("Th", 232): (90, 232.038053606, 7.6150), ("Th", 234): (90, 234.043599801, 7.5969),
    ("U", 235): (92, 235.043928117, 7.5909), ("U", 238): (92, 238.050786936, 7.5701),
}
M_N, M_H = MASS[("n", 1)][1], MASS[("H", 1)][1]      # the neutron, and the hydrogen ATOM (its electron included)
def mass(el, A): return MASS[(el, A)][1]
def z_of(el, A): return MASS[(el, A)][0]

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def closest_approach(Z, K_mev, z=2): return KE2 * z * Z / K_mev                    # d = k(ze)(Ze)/K, in fm
def turn_angle(b, d): return 180.0 if b == 0 else math.degrees(2 * math.atan(d / (2 * abs(b))))   # tan(θ/2) = d/2b
def orbit_energies(r_nm):                                                           # an electron circling one proton, in eV
    pe = -KE2 / r_nm; return (-pe / 2, pe, pe / 2)                                  # (kinetic, potential, total)
def bohr_radius(n=1): return n * n * EPS0 * H * H / (math.pi * M_E * E_CH ** 2)     # r_n = n² a₀, in m
def bohr_speed(n=1): return E_CH ** 2 / (2 * EPS0 * H * n)                          # v_n = v₁/n, in m/s
def bohr_energy(n=1): return -M_E * E_CH ** 4 / (8 * EPS0 ** 2 * H ** 2) / EV / n ** 2   # E_n = −13.6 eV/n²
def photon_nm(e_ev): return HC / e_ev                                               # λ = hc/E
def line_nm(ni, nf): return photon_nm(bohr_energy(ni) - bohr_energy(nf))            # the photon of the jump n_i → n_f
def nuclear_radius(A): return R0 * A ** (1 / 3)                                     # R = R₀ A^(1/3), in fm
def mass_defect(Z, A, M): return Z * M_H + (A - Z) * M_N - M                        # ΔM = Z m_H + N m_n − M, in u
def binding_energy(el, A): return mass_defect(z_of(el, A), A, mass(el, A)) * U_MEV  # E_b = ΔM c², in MeV
def per_nucleon(el, A): return binding_energy(el, A) / A
def q_value(before, after):                                                         # Q = (mass before − mass after) c², in MeV
    return (sum(mass(*x) for x in before) - sum(mass(*x) for x in after)) * U_MEV

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cbrt": lambda x: x ** (1 / 3), "cos": math.cos, "sin": math.sin, "atan": math.atan,
         "tan": math.tan}

def calc(text):
    """Plain arithmetic only: numbers (1.66e-27 too), + − × ÷, powers, brackets, sqrt, cbrt, sin, cos, tan, atan
    (radians) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(fm|nm|pm|mm|cm|m/s|m|MeV|keV|eV|kJ|MJ|J|kWh|kW|MW|W|kg|mg|g|u|N|K|s|days?|years?|%|°|degrees?|"
        r"nucleons?|times|billion years)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'The nuclear atom',
     "q": 'An alpha particle (charge 2e) with 6.0 MeV of kinetic energy heads straight for a copper nucleus (Z = 29). What is its distance of closest approach, in femtometres? Use k e² = 1.44 MeV fm.',
     "parts": ['distance (fm)'],
     "answer": [13.92], "tol": [0.005], "function": '—',
     "hint": 'all the kinetic energy turns into potential energy: K = k(2e)(Ze)/d',
     "working": 'd = k(2e)(Ze)/K = 2 × 29 × 1.44 / 6.0 = 13.92 fm'},
    {"id": 'P1.2', "tier": 1, "group": 'The nuclear atom',
     "q": 'A photon has a wavelength of 500 nm. What is its energy, in electron volts? Use hc = 1240 eV nm.',
     "parts": ['energy (eV)'],
     "answer": [2.48], "tol": [0.005], "function": '—',
     "hint": 'E = 1240 eV nm ÷ the wavelength in nm',
     "working": 'E = hc/λ = 1240 / 500 = 2.48 eV'},
    {"id": 'P1.3', "tier": 1, "group": "Bohr's hydrogen atom",
     "q": "In the Bohr model, what is the radius of hydrogen's n = 2 orbit, in nanometres? (The Bohr radius is 0.0529 nm; answer to three decimals.)",
     "parts": ['radius (nm)'],
     "answer": [0.2116], "tol": [0.0005], "function": '—',
     "hint": 'r_n = n² a₀',
     "working": 'r = n² a₀ = 4 × 0.0529 = 0.2116 nm'},
    {"id": 'P1.4', "tier": 1, "group": "Bohr's hydrogen atom",
     "q": "What is the energy of hydrogen's n = 3 level, in electron volts? (E_n = −13.6 eV/n²; include the sign.)",
     "parts": ['energy (eV)'],
     "answer": [-1.511111111111111], "tol": [0.005], "function": '—',
     "hint": 'E_n = −13.6 eV ÷ n²; a bound level is negative',
     "working": 'E₃ = −13.6 / 9 = −1.51 eV'},
    {"id": 'P1.5', "tier": 1, "group": "Bohr's hydrogen atom",
     "q": "A hydrogen atom drops from the level at −1.51 eV to the level at −3.40 eV. What is the photon's energy, in electron volts?",
     "parts": ['photon energy (eV)'],
     "answer": [1.89], "tol": [0.005], "function": '—',
     "hint": 'the photon carries the gap between the two levels',
     "working": 'E = (−1.51) − (−3.40) = 1.89 eV'},
    {"id": 'P1.6', "tier": 1, "group": 'Inside the nucleus',
     "q": 'What is the radius of a calcium-40 nucleus, in femtometres? Use R = R₀ A^(1/3) with R₀ = 1.2 fm.',
     "parts": ['radius (fm)'],
     "answer": [4.103942272024073], "tol": [0.005], "function": 'cube root',
     "hint": 'the cube root of A, times 1.2 fm (in Python: 40 ** (1/3))',
     "working": 'R = 1.2 × 40^(1/3) = 1.2 × 3.4200 = 4.1039 fm'},
    {"id": 'P1.7', "tier": 1, "group": 'Inside the nucleus',
     "q": 'A nucleus has a mass defect of 0.0500 u. What is its binding energy, in MeV? Use 1 u = 931.5 MeV.',
     "parts": ['binding energy (MeV)'],
     "answer": [46.575], "tol": [0.005], "function": '—',
     "hint": 'multiply the missing mass in u by 931.5 MeV',
     "working": 'E_b = ΔM × 931.5 = 0.0500 × 931.5 = 46.575 MeV'},
    {"id": 'P1.8', "tier": 1, "group": 'Inside the nucleus',
     "q": 'A nucleus of 20 nucleons has a binding energy of 160.6 MeV. What is its binding energy per nucleon, in MeV?',
     "parts": ['binding energy per nucleon (MeV)'],
     "answer": [8.03], "tol": [0.005], "function": '—',
     "hint": 'divide the binding energy by the mass number A',
     "working": '160.6 / 20 = 8.03 MeV per nucleon'},
    {"id": 'P1.9', "tier": 1, "group": 'Nuclear energy',
     "q": "A reaction's products have 0.0200 u less mass than what went in. How much energy is released, in MeV?",
     "parts": ['energy (MeV)'],
     "answer": [18.63], "tol": [0.005], "function": '—',
     "hint": 'the missing mass, times 931.5 MeV per u',
     "working": 'Q = 0.0200 × 931.5 = 18.63 MeV'},
    {"id": 'P1.10', "tier": 1, "group": 'The Sun',
     "q": 'How much energy is 2.0 kg of mass worth, in units of 10¹⁷ joules? Use E = mc² with c = 3.00 × 10⁸ m/s.',
     "parts": ['energy (10¹⁷ J)'],
     "answer": [1.8], "tol": [0.005], "function": '—',
     "hint": 'c² is 9.00 × 10¹⁶ m²/s²',
     "working": 'E = mc² = 2.0 × (3.00 × 10⁸)² = 1.8 × 10¹⁷ J, so 1.80 in units of 10¹⁷ J'},
    {"id": 'P2.1', "tier": 2, "group": 'The nuclear atom',
     "q": 'An alpha particle with 7.7 MeV heads straight for a lead nucleus (Z = 82). What is its distance of closest approach, in femtometres, and what is the electric potential energy of the pair there, in MeV?',
     "parts": ['distance (fm)', 'potential energy (MeV)'],
     "answer": [30.67012987012987, 7.7], "tol": [0.005, 0.005], "function": '—',
     "hint": 'at the closest point the alpha is at rest: K = k(2e)(Ze)/d',
     "working": 'd = 2 × 82 × 1.44 / 7.7 = 30.6701 fm; at that point all 7.7 MeV of kinetic energy is potential energy'},
    {"id": 'P2.2', "tier": 2, "group": 'The nuclear atom',
     "q": 'An electron circles a proton at a radius of 0.4761 nm. What are its kinetic energy, its potential energy and its total energy, in electron volts? Use k e² = 1.44 eV nm.',
     "parts": ['kinetic (eV)', 'potential (eV)', 'total (eV)'],
     "answer": [1.5122873345935726, -3.024574669187145, -1.5122873345935726], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'the pull is the turning force: m v² = k e²/r, so K = k e²/2r, and U = −k e²/r',
     "working": 'U = −k e²/r = −1.44 / 0.4761 = -3.0246 eV; K = −U/2 = 1.5123 eV; E = U/2 = -1.5123 eV'},
    {"id": 'P2.3', "tier": 2, "group": "Bohr's hydrogen atom",
     "q": "For hydrogen's n = 5 orbit, find the radius in nanometres and the electron's speed in units of 10⁵ m/s. (a₀ = 0.0529 nm; v₁ = 2.19 × 10⁶ m/s.)",
     "parts": ['radius (nm)', 'speed (10⁵ m/s)'],
     "answer": [1.3225, 4.38], "tol": [0.005, 0.005], "function": '—',
     "hint": 'r_n = n² a₀ and v_n = v₁/n',
     "working": 'r = n² a₀ = 25 × 0.0529 = 1.3225 nm; v = v₁/n = 2.19 × 10⁶ / 5 = 4.38 × 10⁵ m/s'},
    {"id": 'P2.4', "tier": 2, "group": "Bohr's hydrogen atom",
     "q": "A hydrogen atom drops from n = 5 to n = 2. What is the photon's energy in electron volts, and its wavelength in nanometres? Use E_n = −13.6 eV/n² and hc = 1240 eV nm.",
     "parts": ['photon energy (eV)', 'wavelength (nm)'],
     "answer": [2.856, 434.17366946778714], "tol": [0.005, 0.05], "function": '—',
     "hint": 'the gap between the two levels; then λ = 1240 ÷ E',
     "working": 'E = 13.6 × (1/4 − 1/25) = 2.8560 eV; λ = 1240 / 2.8560 = 434.17 nm (the violet Balmer line)'},
    {"id": 'P2.5', "tier": 2, "group": "Bohr's hydrogen atom",
     "q": 'How much energy, in electron volts, does it take to lift a hydrogen atom from its ground state to n = 3? And how much to free the electron completely from the ground state?',
     "parts": ['to n = 3 (eV)', 'to free it (eV)'],
     "answer": [12.088888888888889, 13.6], "tol": [0.005, 0.005], "function": '—',
     "hint": 'a free electron at rest has zero energy',
     "working": 'E₃ − E₁ = −1.51 − (−13.6) = 12.0889 eV; to free it: 0 − (−13.6) = 13.6 eV (the ionisation energy)'},
    {"id": 'P2.6', "tier": 2, "group": 'Inside the nucleus',
     "q": 'An iron-56 atom has a mass of 55.934936 u. Its 26 hydrogen atoms weigh 1.007825 u each and its 30 neutrons 1.008665 u each. What is the mass defect in u (to four decimals), the binding energy in MeV (to the nearest MeV) and the binding energy per nucleon in MeV?',
     "parts": ['mass defect (u)', 'binding energy (MeV)', 'per nucleon (MeV)'],
     "answer": [0.5284639999999925, 492.264215999993, 8.790432428571304], "tol": [5e-05, 0.5, 0.005], "function": '—',
     "hint": 'ΔM = Z m_H + N m_n − M; then × 931.5; then ÷ A',
     "working": 'ΔM = 26 × 1.007825 + 30 × 1.008665 − 55.934936 = 0.528464 u; E_b = ΔM × 931.5 = 492.26 MeV; per nucleon: ÷ 56 = 8.7904 MeV'},
    {"id": 'P2.7', "tier": 2, "group": 'Inside the nucleus',
     "q": 'What are the radii of a helium-4 nucleus and of a lead-208 nucleus, in femtometres? (R₀ = 1.2 fm.)',
     "parts": ['helium-4 (fm)', 'lead-208 (fm)'],
     "answer": [1.9048812623618392, 7.109990564177688], "tol": [0.005, 0.005], "function": 'cube root',
     "hint": 'R = R₀ A^(1/3)',
     "working": 'R(He) = 1.2 × 4^(1/3) = 1.9049 fm; R(Pb) = 1.2 × 208^(1/3) = 7.1100 fm: 52 times the nucleons, under 4 times the radius'},
    {"id": 'P2.8', "tier": 2, "group": 'Nuclear energy',
     "q": 'Uranium-238 (238.050787 u) decays to thorium-234 (234.043600 u) and an alpha particle (helium-4 atom, 4.002603 u). What mass goes missing, in u (to five decimals), and how much energy is released, in MeV?',
     "parts": ['missing mass (u)', 'energy (MeV)'],
     "answer": [0.004584000000016353, 4.269996000015233], "tol": [5e-06, 0.005], "function": '—',
     "hint": 'mass before minus mass after; then × 931.5 MeV',
     "working": 'Δm = 238.050787 − 234.043600 − 4.002603 = 0.004584 u; Q = Δm × 931.5 = 4.2700 MeV'},
    {"id": 'P2.9', "tier": 2, "group": 'Nuclear energy',
     "q": 'Deuterium (2.014102 u) and tritium (3.016049 u) fuse into helium-4 (4.002603 u) and a neutron (1.008665 u). How much energy is released, in MeV?',
     "parts": ['energy (MeV)'],
     "answer": [17.589514500000398], "tol": [0.005], "function": '—',
     "hint": 'mass before minus mass after; then × 931.5 MeV',
     "working": 'Δm = 2.014102 + 3.016049 − 4.002603 − 1.008665 = 0.018883 u; Q = 0.018883 × 931.5 = 17.5895 MeV'},
    {"id": 'P2.10', "tier": 2, "group": 'The Sun',
     "q": 'A star shines with a power of 9.0 × 10²⁶ W. How much mass does it turn into light each second, in units of 10⁹ kg? (c = 3.00 × 10⁸ m/s.)',
     "parts": ['mass a second (10⁹ kg)'],
     "answer": [10.0], "tol": [0.005], "function": '—',
     "hint": 'E = mc², run for one second',
     "working": 'm = E/c² each second = 9.0 × 10²⁶ / (9.00 × 10¹⁶) = 1.0 × 10¹⁰ kg = 10.00 in units of 10⁹ kg'},
    {"id": 'P3.1', "tier": 3, "group": 'The nuclear atom',
     "q": 'An alpha particle must come within 8.0 fm of the centre of a uranium nucleus (Z = 92), head-on. What kinetic energy does it need, in MeV?',
     "parts": ['kinetic energy (MeV)'],
     "answer": [33.12], "tol": [0.005], "function": '—',
     "hint": 'the closest-approach formula, run backwards',
     "working": 'K = k(2e)(Ze)/d = 2 × 92 × 1.44 / 8.0 = 33.12 MeV'},
    {"id": 'P3.2', "tier": 3, "group": "Bohr's hydrogen atom",
     "q": "A line of hydrogen's Balmer series (jumps that end on n = 2) has a wavelength of 397.0 nm. From which level did the electron drop? (Give the whole number.)",
     "parts": ['level n'],
     "answer": [7.0], "tol": [0.05], "function": '—',
     "hint": "the photon's energy first; then solve 13.6 (1/4 − 1/n²) for n",
     "working": 'E = 1240 / 397.0 = 3.1234 eV = 13.6 × (1/4 − 1/n²), so 1/n² = 0.25 − 0.22966 and n = 7.01: level 7'},
    {"id": 'P3.3', "tier": 3, "group": "Bohr's hydrogen atom",
     "q": "An electron is in hydrogen's n = 3 orbit. What is its de Broglie wavelength, in nanometres, and how many of those wavelengths fit round the orbit? (r₃ = 0.4761 nm.)",
     "parts": ['wavelength (nm)', 'wavelengths round the orbit'],
     "answer": [0.9971415082494004, 3.0], "tol": [0.005, 0.005], "function": 'pi',
     "hint": "de Broglie's condition: n λ = 2π r",
     "working": 'n wavelengths fit round the orbit: 3 λ = 2π r, so λ = 2π × 0.4761 / 3 = 0.9971 nm'},
    {"id": 'P3.4', "tier": 3, "group": 'Inside the nucleus',
     "q": 'A nucleus has a radius of 4.8 fm. About how many nucleons does it hold? (R₀ = 1.2 fm.) And by what factor does the radius grow if the number of nucleons is made 27 times bigger?',
     "parts": ['nucleons', 'radius factor'],
     "answer": [64.0, 3.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'R = R₀ A^(1/3), run backwards: A = (R/R₀)³',
     "working": 'A = (R/R₀)³ = (4.8/1.2)³ = 4³ = 64; 27 times the nucleons: the cube root of 27 = 3 times the radius'},
    {"id": 'P3.5', "tier": 3, "group": 'Inside the nucleus',
     "q": 'Lithium-7 (7.016003 u) has 3 protons and 4 neutrons. Find its binding energy per nucleon in MeV. Then say how much energy, in MeV, is released for each nucleon when nuclei like it end up as helium-4 (7.07 MeV per nucleon).',
     "parts": ['per nucleon (MeV)', 'released per nucleon (MeV)'],
     "answer": [5.606565428571377, 1.463434571428623], "tol": [0.005, 0.005], "function": '—',
     "hint": 'a nucleus higher on the curve is more tightly bound: the difference is released',
     "working": 'ΔM = 3 × 1.007825 + 4 × 1.008665 − 7.016003 = 0.042132 u; E_b = 39.2460 MeV; per nucleon 5.6066 MeV; climbing to 7.07 releases 1.4634 MeV for each nucleon'},
    {"id": 'P3.6', "tier": 3, "group": 'Nuclear energy',
     "q": "Each fission of uranium-235 (235.043928 u) releases about 200 MeV in all. How much energy does 1.0 kg release, in units of 10¹³ J? And how many days would that run a 1.0 × 10⁹ W power station's reactor?",
     "parts": ['energy (10¹³ J)', 'days'],
     "answer": [8.209982954616894, 0.9502295086362146], "tol": [0.005, 0.005], "function": '—',
     "hint": 'count the nuclei in a kilogram; 1 MeV = 1.602 × 10⁻¹³ J; a watt is a joule a second',
     "working": 'nuclei = 1000 / 235.043928 × 6.022 × 10²³ = 2.5621e+24; energy = × 200 MeV × 1.602 × 10⁻¹³ J = 8.2100e+13 J; time = 8.210e+13 / 10⁹ W = 82100 s = 0.9502 days'},
    {"id": 'P3.7', "tier": 3, "group": 'The Sun',
     "q": "The Sun's power is 3.828 × 10²⁶ W, and each helium nucleus it makes releases 26.73 MeV. How many helium nuclei does it make each second, in units of 10³⁷? And how much hydrogen does that use each second, in units of 10¹¹ kg? (Four hydrogen atoms of 1.007825 u each; 1 u = 1.66054 × 10⁻²⁷ kg.)",
     "parts": ['helium nuclei a second (10³⁷)', 'hydrogen a second (10¹¹ kg)'],
     "answer": [8.9384574399685, 5.9835239918934695], "tol": [0.005, 0.005], "function": '—',
     "hint": 'power ÷ the energy of one reaction; then four hydrogen atoms for each',
     "working": 'rate = 3.828 × 10²⁶ / (26.73 × 1.602 × 10⁻¹³) = 8.9385e+37 a second; hydrogen = rate × 4 × 1.007825 × 1.66054 × 10⁻²⁷ = 5.9835e+11 kg a second: about 600 million tonnes'},
    {"id": 'P3.8', "tier": 3, "group": 'The Sun',
     "q": "A star has twice the Sun's mass (4.0 × 10³⁰ kg) and shines four times as brightly, so it uses hydrogen four times as fast as the Sun's 5.98 × 10¹¹ kg a second. If it can use a tenth of its mass, how long does it last, in billions of years? (A year is 3.156 × 10⁷ s.)",
     "parts": ['lifetime (billion years)'],
     "answer": [5.2990113401386205], "tol": [0.005], "function": '—',
     "hint": 'time = the usable mass ÷ the rate; then seconds to years',
     "working": 'usable hydrogen = 0.1 × 4.0 × 10³⁰ = 4.0 × 10²⁹ kg; rate = 4 × 5.98 × 10¹¹ = 2.392 × 10¹² kg/s; time = 1.6722e+17 s = 5.2990 billion years: bigger stars burn out sooner'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert round(K / 1e9, 2) == 8.99 and round(HC) == 1240 and round(KE2, 2) == 1.44 and close(U_KG * C * C / MEV, U_MEV, 1e-9)
print("Ready: k e² =", round(KE2, 4), "MeV fm; h c =", round(HC, 2), "eV nm; 1 u =", round(U_MEV, 2), "MeV")

## Tier 1 · Warm-up

One idea, one step: a distance of closest approach, a photon's energy, a Bohr orbit's radius, an energy level, the
photon of a jump, a nucleus's radius, a binding energy, a binding energy per nucleon, a reaction's energy, and the
energy that a mass is worth.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `closest_approach(Z, K_mev)`,
`orbit_energies(r_nm)`, `bohr_radius(n)`, `bohr_speed(n)`, `bohr_energy(n)`, `line_nm(ni, nf)`, `nuclear_radius(A)`,
`mass_defect(Z, A, M)`, `binding_energy("He", 4)`, `q_value(before, after)`. Two handy numbers do most of the work by
hand: k e² = 1.44 MeV fm (or eV nm) and h c = 1240 eV nm. Each line below computes one warm-up answer and checks it
against the practice set (the exercises use the rounded values the course says: 1.44, 1240, 13.6, 0.0529, 931.5).

In [ ]:
print(f"P1.1  d = {2 * 29 * 1.44 / 6.0:.2f} fm;  P1.6  R = {1.2 * 40 ** (1 / 3):.2f} fm")
assert agrees("P1.1", [2 * 29 * 1.44 / 6.0]) and agrees("P1.2", [1240 / 500]) and agrees("P1.3", [4 * 0.0529])
assert agrees("P1.4", [-13.6 / 3 ** 2]) and agrees("P1.5", [-1.51 - (-3.40)]) and agrees("P1.6", [1.2 * 40 ** (1 / 3)])
assert agrees("P1.7", [0.0500 * 931.5]) and agrees("P1.8", [160.6 / 20]) and agrees("P1.9", [0.0200 * 931.5])
assert agrees("P1.10", [2.0 * (3.00e8) ** 2 / 1e17])
# the full constants agree with the rounded ones the exercises use
assert close(closest_approach(29, 6.0), 2 * 29 * 1.44 / 6.0, 1e-3) and close(bohr_radius(2) * 1e9, 4 * 0.0529, 1e-3)
assert close(bohr_energy(3), -13.6 / 9, 1e-3)
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## Alpha particles past a nucleus

Each alpha particle is stepped a little at a time (fourth-order Runge–Kutta) under the electric push of the nucleus,
k (2e)(Ze)/r². Nothing about the paths is drawn by hand. An alpha aimed far from the nucleus hardly turns; one aimed
close swings wide; the one aimed straight at it stops at the **distance of closest approach**, d = 2kZe²/K, and comes
straight back. Raise the alpha's energy and d shrinks: the alpha probes deeper. Choose a heavier target and every path
bends more. The turning angle of each path is checked against the formula tan(θ/2) = d/2b, where b is how far the aim
misses the centre.

In [ ]:
def step_many(bs, x0=-2000.0, eta=0.04, keep=False):
    """Alphas aimed along +x with impact parameters bs, all stepped at once. Lengths are in units of d (the head-on
    closest distance) and the far-away speed is 1, so the acceleration is r̂/(2r²). Each alpha's step is eta × its
    distance. Returns the turning angles in degrees, the closest distances, and (if keep) the paths."""
    b = np.asarray(bs, float); x = np.full_like(b, x0); y = b.copy(); vx = np.ones_like(b); vy = np.zeros_like(b)
    rmin = np.hypot(x, y); live = np.ones(b.shape, bool); path = [(x.copy(), y.copy())]
    def f(x, y):
        r2 = x * x + y * y; r3 = 2 * r2 * np.sqrt(r2)
        return x / r3, y / r3
    for _ in range(20000):
        if not live.any(): break
        h = np.where(live, eta * np.hypot(x, y), 0.0)
        a1 = f(x, y)
        x2, y2, u2, w2 = x + h / 2 * vx, y + h / 2 * vy, vx + h / 2 * a1[0], vy + h / 2 * a1[1]; a2 = f(x2, y2)
        x3, y3, u3, w3 = x + h / 2 * u2, y + h / 2 * w2, vx + h / 2 * a2[0], vy + h / 2 * a2[1]; a3 = f(x3, y3)
        x4, y4, u4, w4 = x + h * u3, y + h * w3, vx + h * a3[0], vy + h * a3[1]; a4 = f(x4, y4)
        x = x + h / 6 * (vx + 2 * u2 + 2 * u3 + u4); y = y + h / 6 * (vy + 2 * w2 + 2 * w3 + w4)
        vx = vx + h / 6 * (a1[0] + 2 * a2[0] + 2 * a3[0] + a4[0]); vy = vy + h / 6 * (a1[1] + 2 * a2[1] + 2 * a3[1] + a4[1])
        rr = np.hypot(x, y); rmin = np.minimum(rmin, rr)
        live &= ~((rr > abs(x0)) & (x * vx + y * vy > 0))
        if keep: path.append((x.copy(), y.copy()))
    return np.degrees(np.arctan2(vy, vx)), rmin, path

AIMS = np.array([0.0, 0.05, 0.15, 0.3, 0.5, 0.8, 1.2, 2.0, 3.5])     # impact parameters, in units of d

def alpha_picture(energy_MeV=5.3, Z=79):
    d = closest_approach(Z, energy_MeV)                              # fm
    bs = np.concatenate([AIMS, -AIMS[1:]])
    ang, rmin, path = step_many(bs, x0=-1000.0, eta=0.01, keep=True)
    fig, ax = plt.subplots(figsize=(10, 6)); fig.subplots_adjust(left=0.09, right=0.98, top=0.88, bottom=0.12)
    xs = np.array([p[0] for p in path]) * d; ys = np.array([p[1] for p in path]) * d
    for k in range(len(bs)):
        ax.plot(xs[:, k], ys[:, k], color=GOLD if bs[k] == 0 else CHALK, lw=2.6 if bs[k] == 0 else 1.6)
    ax.plot([0], [0], "o", color=CORAL, ms=9)
    span = 8 * d
    ax.set_xlim(-span, span); ax.set_ylim(-span * 0.62, span * 0.62); ax.set_aspect("equal")
    ax.set_xlabel("distance (fm)"); ax.set_ylabel("distance (fm)")
    fig.suptitle(f"closest approach d = 2kZe²/K = {d:.1f} fm   ·   the head-on alpha turns at {rmin[0] * d:.1f} fm",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "each path stepped (Runge-Kutta) · the gold path is aimed straight at the nucleus"); plt.show()
    return d, rmin[0] * d

d_gold, turn_gold = alpha_picture(5.3, 79)
assert round(d_gold) == 43 and close(turn_gold, d_gold, 2e-3)                      # 43 fm, and the simulated alpha stops there
far_angles, far_rmin, _ = step_many(AIMS)                                          # started far away, for the check
assert all(abs(a - turn_angle(b, 1.0)) < 0.1 for a, b in zip(far_angles, AIMS))    # every angle: tan(θ/2) = d/2b
assert abs(far_rmin[0] - 1) < 2e-3 and round(turn_angle(0.5, 1.0)) == 90           # head-on: stops at d; b = d/2: 90°
assert close(closest_approach(79, 10.6), d_gold / 2, 1e-9)                         # twice the energy: half the distance

In [ ]:
interact(alpha_picture, energy_MeV=FloatSlider(value=5.3, min=2.0, max=30.0, step=0.1, description="alpha (MeV)"),
         Z=IntSlider(value=79, min=13, max=92, step=1, description="target Z"));

## Counting by angle

Now many alphas, aimed evenly over a disc round one nucleus (out to ten times d). Most are aimed far out and hardly
turn; a few are aimed close and turn through large angles. The bars count how many end up in each range of angle.
The gold line is Rutherford's formula for the same count: every alpha aimed inside b = (d/2) cot(θ/2) turns by more
than θ. The counts follow it at every angle. Per unit of solid angle (what a small detector sees as it swings round
the foil) the same formula reads 1/sin⁴(θ/2).

In [ ]:
EDGES = [10, 20, 30, 45, 60, 90, 120, 150, 180]

def beyond(theta_deg, n, b_max=10.0):
    """Rutherford's count turned by more than θ, of n alphas aimed evenly within b_max (in units of d)."""
    return n * (0.5 / math.tan(math.radians(theta_deg) / 2) / b_max) ** 2

def count_picture(alphas=10000):
    bs = 10.0 * np.sqrt((np.arange(alphas) + 0.5) / alphas)                        # evenly over the disc's area
    ang, _, _ = step_many(bs)
    bins = [int(np.sum((ang >= lo) & (ang < hi + (1e-9 if hi == 180 else 0)))) for lo, hi in zip(EDGES, EDGES[1:])]
    want = [beyond(lo, alphas) - (beyond(hi, alphas) if hi < 180 else 0.0) for lo, hi in zip(EDGES, EDGES[1:])]
    fig, ax = plt.subplots(figsize=(10, 5)); fig.subplots_adjust(left=0.1, right=0.98, top=0.88, bottom=0.13)
    mids = range(len(bins))
    ax.bar(mids, bins, color=CHALK, width=0.7); ax.plot(mids, want, "o-", color=GOLD, lw=1.6, ms=5)
    ax.set_yscale("log"); ax.set_xticks(list(mids)); ax.set_xticklabels([f"{lo}°–{hi}°" for lo, hi in zip(EDGES, EDGES[1:])])
    ax.set_ylabel("alphas (log scale)"); ax.set_xlabel("turned through")
    back = int(np.sum(ang > 90))
    fig.suptitle(f"{alphas:,} alphas · {back} turned back by more than 90° · formula: {beyond(90, alphas):.1f}",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "bars: counted · gold: Rutherford's formula"); plt.show()
    return ang, bins, want

ang, bins, want = count_picture(10000)
assert int(np.sum(ang > 90)) == 25 and close(beyond(90, 10000), 25, 1e-9)          # 1 in 400 of these turn back
assert all(abs(n - w) <= max(2, 2e-3 * w) for n, w in zip(bins, want))             # every bar on the formula
ratio = (1 / math.sin(math.radians(15)) ** 4) / (1 / math.sin(math.radians(30)) ** 4)
assert round(ratio) == 14                                                          # 1/sin⁴(θ/2): 14 times more at 30° than at 60°

In [ ]:
interact(count_picture, alphas=IntSlider(value=10000, min=2000, max=40000, step=2000, description="alphas"));

## The pudding atom

Before Rutherford, one guess spread the atom's positive charge evenly through the whole atom. Here the same 5.3 MeV
alpha crosses such a ball of charge (the size of a gold atom, with gold's 79 charges). Inside, the push grows in step
with the distance from the centre; outside, the electrons cancel it. The bars show the turning angle for each aim. It
is never more than about a hundredth of a degree: a pudding atom cannot send an alpha back.

In [ ]:
def pudding_turn(b, eps, steps=1200):
    """An alpha through a ball of charge of radius 1: the push inside is (eps/2) × the distance from the centre.
    Stepped (Runge-Kutta); returns the turning angle in degrees. eps = d/R."""
    def f(s):
        k = eps / 2 if s[0] * s[0] + s[1] * s[1] < 1.0 else 0.0
        return (s[2], s[3], k * s[0], k * s[1])
    s, h = (-1.2, b, 1.0, 0.0), 2.4 / steps
    for _ in range(steps):
        k1 = f(s)
        k2 = f(tuple(a + h / 2 * q for a, q in zip(s, k1)))
        k3 = f(tuple(a + h / 2 * q for a, q in zip(s, k2)))
        k4 = f(tuple(a + h * q for a, q in zip(s, k3)))
        s = tuple(a + h / 6 * (q1 + 2 * q2 + 2 * q3 + q4) for a, q1, q2, q3, q4 in zip(s, k1, k2, k3, k4))
    return math.degrees(math.atan2(s[3], s[2]))

def pudding_picture(energy_MeV=5.3, Z=79, atom_pm=144):
    eps = closest_approach(Z, energy_MeV) * 1e-15 / (atom_pm * 1e-12)
    bs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7071, 0.8, 0.9]
    turns = [pudding_turn(b, eps) for b in bs]
    fig, ax = plt.subplots(figsize=(10, 4.6)); fig.subplots_adjust(left=0.11, right=0.98, top=0.86, bottom=0.15)
    ax.bar([str(round(b, 2)) for b in bs], turns, color=CHALK, width=0.6)
    ax.set_xlabel("aim: distance from the centre, as a fraction of the atom's radius"); ax.set_ylabel("turn (degrees)")
    fig.suptitle(f"the greatest turn: {max(turns):.4f}°   ·   bouncing back needs 180°", x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return eps, max(turns)

eps, worst = pudding_picture()
assert worst < 0.01 and close(math.radians(worst), eps / 2, 5e-3)                  # at most ε/2: under a hundredth of a degree

In [ ]:
interact(pudding_picture, energy_MeV=FloatSlider(value=5.3, min=2.0, max=10.0, step=0.1, description="alpha (MeV)"),
         Z=IntSlider(value=79, min=13, max=92, step=1, description="charges Z"),
         atom_pm=IntSlider(value=144, min=50, max=250, step=2, description="atom (pm)"));

## Tier 2 · Core

The level of the course's worked problems: closest approach with its energy, an orbiting electron's three energies, a
Bohr orbit's radius and speed, a Balmer line, the energy to lift and to free an electron, iron's binding energy, two
nuclear radii, an alpha decay, a fusion reaction, and a star's mass turning into light.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

For an orbit, find the potential energy first: the kinetic energy is half its size and positive, and the total is half
of it and negative. For a line, find the gap between the two levels, then divide 1240 by it. For a nucleus, add up the
hydrogen atoms and the neutrons, subtract the atom's mass, and multiply by 931.5.

In [ ]:
assert agrees("P2.1", [2 * 82 * 1.44 / 7.7, 7.7])
U = -1.44 / 0.4761
assert agrees("P2.2", [-U / 2, U, U / 2]) and agrees("P2.3", [25 * 0.0529, 2.19e6 / 5 / 1e5])
gap = 13.6 * (1 / 4 - 1 / 25)
assert agrees("P2.4", [gap, 1240 / gap]) and agrees("P2.5", [13.6 - 13.6 / 9, 13.6])
dM = 26 * 1.007825 + 30 * 1.008665 - 55.934936
print(f"P2.6  ΔM = {dM:.4f} u, E_b = {dM * 931.5:.1f} MeV, per nucleon {dM * 931.5 / 56:.2f} MeV")
assert agrees("P2.6", [dM, dM * 931.5, dM * 931.5 / 56]) and agrees("P2.7", [1.2 * 4 ** (1 / 3), 1.2 * 208 ** (1 / 3)])
dm = 238.050787 - 234.043600 - 4.002603
assert agrees("P2.8", [dm, dm * 931.5]) and agrees("P2.9", [(2.014102 + 3.016049 - 4.002603 - 1.008665) * 931.5])
assert agrees("P2.10", [9.0e26 / (3.00e8) ** 2 / 1e9])
# the mass table gives the same answers as the exercises' rounded masses
assert abs(per_nucleon("Fe", 56) - dM * 931.5 / 56) < 0.01 and abs(q_value([("U", 238)], [("Th", 234), ("He", 4)]) - dm * 931.5) < 0.01
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## Bohr's orbits, levels and lines

Bohr's two equations (the electric pull is the turning force; m v r = n h/2π) give the allowed radii r = n² a₀, the
speeds v = v₁/n and the energies E = −13.6 eV/n². Here they are worked out from the constants alone. Choose a jump: the
picture shows the two levels, the photon's energy (the gap between them) and its wavelength, 1240 eV nm ÷ the gap. The
four jumps that end on level 2 land on the four lines of hydrogen that we can see; their measured wavelengths are
marked beside the computed ones. Jumps that end on level 1 are in the ultraviolet; jumps that end on level 3, in the
infrared.

In [ ]:
MEASURED = {(3, 2): 656.279, (4, 2): 486.135, (5, 2): 434.047, (6, 2): 410.174}    # nm, in air (NIST)

def rgb(nm):
    """An approximate screen colour for a wavelength of visible light (CHALK outside it)."""
    if nm < 380 or nm > 750: return CHALK
    if nm < 440: c = ((440 - nm) / 60, 0, 1)
    elif nm < 490: c = (0, (nm - 440) / 50, 1)
    elif nm < 510: c = (0, 1, (510 - nm) / 20)
    elif nm < 580: c = ((nm - 510) / 70, 1, 0)
    elif nm < 645: c = (1, (645 - nm) / 65, 0)
    else: c = (1, 0, 0)
    return tuple(0.25 + 0.75 * v for v in c)

def jump_picture(n_from=3, n_to=2):
    if n_from <= n_to: n_from = n_to + 1
    gap = bohr_energy(n_from) - bohr_energy(n_to); nm = photon_nm(gap)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.2), gridspec_kw={"width_ratios": [1, 1.5]})
    fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.14, wspace=0.22)
    for n in range(1, 8):
        a1.hlines(bohr_energy(n), 0, 1, color=CHALK if n in (n_from, n_to) else DIM, lw=2 if n in (n_from, n_to) else 1.2)
        if n <= 4: a1.text(1.03, bohr_energy(n), f"n = {n}  {bohr_energy(n):.2f} eV", va="center", fontsize=11, color=DIM)
    a1.hlines(0, 0, 1, color=DIM, ls="--", lw=1)
    a1.annotate("", xy=(0.5, bohr_energy(n_to)), xytext=(0.5, bohr_energy(n_from)), arrowprops=dict(color=VIOLET, lw=2.4, arrowstyle="->"))
    a1.set_xlim(0, 1.75); a1.set_xticks([]); a1.set_ylabel("energy (eV)"); a1.grid(False)
    a2.axvspan(380, 750, color=GHOST, alpha=0.5)
    for (ni, nf), m in MEASURED.items(): a2.vlines(m, 0, 0.45, color=rgb(m), lw=3)
    a2.vlines(nm, 0.55, 1.0, color=GOLD if not 380 < nm < 750 else rgb(nm), lw=3)
    a2.set_xscale("log"); a2.set_xlim(85, 2100); a2.set_ylim(0, 1.05); a2.set_yticks([0.22, 0.78]); a2.set_yticklabels(["measured\n(Balmer)", "computed"])
    a2.set_xticks([100, 200, 400, 700, 1000, 2000]); a2.set_xticklabels(["100", "200", "400", "700", "1000", "2000"])
    a2.set_xlabel("wavelength (nm, log scale) · the shaded band is visible light"); a2.grid(False)
    fig.suptitle(f"jump {n_from} → {n_to}:  gap {gap:.3f} eV,  wavelength {nm:.1f} nm", x=0.03, ha="left", color=GOLD)
    footnote(fig, "levels from the constants · measured lines: NIST, in air"); plt.show()
    return gap, nm

gap32, nm32 = jump_picture(3, 2)
assert round(bohr_radius(1) * 1e9, 4) == 0.0529 and round(bohr_speed(1) / 1e6, 2) == 2.19 and round(bohr_energy(1), 1) == -13.6
assert round(gap32, 2) == 1.89 and round(nm32) == 656
for (ni, nf), m in MEASURED.items():
    assert abs(line_nm(ni, nf) / m - 1) < 3e-4                                    # Bohr's lines: within 0.03 % of the measured ones
for n in (1, 2, 3, 4):                                                             # both of Bohr's equations hold on every orbit
    rn, vn = bohr_radius(n), bohr_speed(n)
    assert close(M_E * vn ** 2 / rn, K * E_CH ** 2 / rn ** 2, 1e-9) and close(M_E * vn * rn, n * H / (2 * math.pi), 1e-9)
    assert close(2 * math.pi * rn / (H / (M_E * vn)), n, 1e-9)                    # de Broglie: n whole wavelengths fit round it
assert round(line_nm(2, 1)) == 122 and round(line_nm(4, 3)) == 1875                # ultraviolet, and infrared

In [ ]:
interact(jump_picture, n_from=IntSlider(value=3, min=2, max=7, step=1, description="from level"),
         n_to=IntSlider(value=2, min=1, max=4, step=1, description="to level"));

## The binding-energy curve

For each nucleus in the mass table: add up its hydrogen atoms and its neutrons, subtract the atom's measured mass
(that is the mass defect), multiply by 931.5 MeV, and divide by the number of nucleons. Plotted against the mass
number, the points make the binding-energy curve. It climbs steeply, peaks near iron and nickel at about 8.8 MeV, and
falls slowly to 7.6 MeV at uranium. Choose a nucleus to see its numbers. A nucleus higher on the curve is more tightly
bound: that is why splitting the heaviest nuclei, or joining the lightest, releases energy.

In [ ]:
NAMES = sorted((f"{el}-{A}", (el, A)) for (el, A) in MASS if A >= 2)

def curve_picture(nucleus="Fe-56"):
    el, A = dict(NAMES)[nucleus]
    pts = sorted((a, per_nucleon(e, a)) for (e, a) in MASS if a >= 2)
    fig, ax = plt.subplots(figsize=(10, 5.4)); fig.subplots_adjust(left=0.09, right=0.98, top=0.86, bottom=0.13)
    ax.plot([p[0] for p in pts], [p[1] for p in pts], "-", color=DIM, lw=1.2)
    ax.plot([p[0] for p in pts], [p[1] for p in pts], "o", color=CHALK, ms=4.5)
    ax.plot([A], [per_nucleon(el, A)], "o", color=GOLD, ms=11)
    ax.set_xlabel("mass number A"); ax.set_ylabel("binding energy per nucleon (MeV)"); ax.set_ylim(0, 9.5)
    Z = z_of(el, A); dM = mass_defect(Z, A, mass(el, A))
    fig.suptitle(f"{nucleus}:  ΔM = {dM:.4f} u,  E_b = {dM * U_MEV:.1f} MeV,  {per_nucleon(el, A):.2f} MeV a nucleon",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "from atomic masses: AME2020"); plt.show()
    return per_nucleon(el, A)

fe = curve_picture("Fe-56")
assert round(fe, 2) == 8.79 and round(per_nucleon("He", 4), 1) == 7.1 and round(per_nucleon("U", 238), 1) == 7.6
assert max(((per_nucleon(e, a), e, a) for (e, a) in MASS if a >= 2))[1:] == ("Ni", 62)      # the very top: nickel-62
assert all(abs(per_nucleon(e, a) - MASS[(e, a)][2]) < 1e-4 for (e, a) in MASS if a >= 2)    # each one: the evaluation's own figure
assert round(binding_energy("He", 4), 1) == 28.3 and round(mass_defect(2, 4, mass("He", 4)), 4) == 0.0304

In [ ]:
interact(curve_picture, nucleus=Dropdown(options=[n for n, _ in NAMES], value="Fe-56", description="nucleus"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: the energy an alpha needs, the level a line came from, an
electron's wavelength in its orbit, a nucleus's nucleons from its size, lithium against helium on the curve, a kilogram
of uranium, the Sun's helium and hydrogen each second, and a bigger star's lifetime.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Run backwards, a formula is still one line: K = 2Z × 1.44/d for the energy; 1/n² = 1/4 − E/13.6 for the level;
A = (R/R₀)³ for the nucleons. For a mass of fuel, count the nuclei first (the mass in grams ÷ the atomic mass, times
6.022 × 10²³), then multiply by the energy of one reaction, with 1 MeV = 1.602 × 10⁻¹³ J.

In [ ]:
assert agrees("P3.1", [2 * 92 * 1.44 / 8.0])
E = 1240 / 397.0; n = (1 / (1 / 4 - E / 13.6)) ** 0.5
print(f"P3.2  E = {E:.3f} eV, n = {n:.2f}")
assert agrees("P3.2", [round(n)]) and agrees("P3.3", [2 * math.pi * 0.4761 / 3, 3]) and agrees("P3.4", [(4.8 / 1.2) ** 3, 27 ** (1 / 3)])
li = (3 * 1.007825 + 4 * 1.008665 - 7.016003) * 931.5 / 7
assert agrees("P3.5", [li, 7.07 - li])
nuclei = 1000.0 / 235.043928 * N_A; joules = nuclei * 200 * MEV
assert agrees("P3.6", [joules / 1e13, joules / 1e9 / 86400])
rate = 3.828e26 / (26.73 * MEV)
assert agrees("P3.7", [rate / 1e37, rate * 4 * 1.007825 * 1.66054e-27 / 1e11])
assert agrees("P3.8", [0.1 * 4.0e30 / (4 * 5.98e11) / YEAR / 1e9])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## The energy of a nuclear reaction

Whatever the reaction, the rule is the same: add up the masses that go in, add up the masses that come out, and the
difference, times 931.5 MeV for each atomic mass unit, is the energy released. No particles are destroyed: the
nucleons are counted before and after. Choose a reaction. The masses are whole atoms' masses from the table, so the
electrons cancel (the first step of the Sun's chain makes a positron, which is why it is listed with its own sum).

In [ ]:
REACTIONS = {
    "fission: U-235 + n → Ba-141 + Kr-92 + 3n": ([("U", 235), ("n", 1)], [("Ba", 141), ("Kr", 92), ("n", 1), ("n", 1), ("n", 1)]),
    "fission: U-235 + n → Xe-140 + Sr-94 + 2n": ([("U", 235), ("n", 1)], [("Xe", 140), ("Sr", 94), ("n", 1), ("n", 1)]),
    "fusion in the Sun, in all: 4 H-1 → He-4": ([("H", 1)] * 4, [("He", 4)]),
    "fusion: H-2 + H-3 → He-4 + n": ([("H", 2), ("H", 3)], [("He", 4), ("n", 1)]),
    "fusion: H-2 + H-2 → He-3 + n": ([("H", 2), ("H", 2)], [("He", 3), ("n", 1)]),
    "alpha decay: Po-210 → Pb-206 + He-4": ([("Po", 210)], [("Pb", 206), ("He", 4)]),
    "alpha decay: Ra-226 → Rn-222 + He-4": ([("Ra", 226)], [("Rn", 222), ("He", 4)]),
    "alpha decay: U-238 → Th-234 + He-4": ([("U", 238)], [("Th", 234), ("He", 4)]),
    "beta decay: C-14 → N-14": ([("C", 14)], [("N", 14)]),
}

def reaction_picture(reaction="fission: U-235 + n → Ba-141 + Kr-92 + 3n"):
    before, after = REACTIONS[reaction]
    m0, m1 = sum(mass(*x) for x in before), sum(mass(*x) for x in after); Q = (m0 - m1) * U_MEV
    A0, A1 = sum(a for _, a in before), sum(a for _, a in after)
    fig, ax = plt.subplots(figsize=(10, 3.8)); fig.subplots_adjust(left=0.12, right=0.97, top=0.8, bottom=0.2)
    ax.barh(["after", "before"], [m1 - m1, m0 - m1], left=0, color=[CHALK, VIOLET], height=0.5)
    ax.set_xlabel("mass above the products' mass (u)"); ax.grid(False)
    fig.suptitle(f"missing mass {m0 - m1:.6f} u  →  {Q:.2f} MeV   ·   nucleons: {A0} before, {A1} after",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "atomic masses: AME2020"); plt.show()
    return Q, A0, A1

Q_fis, A0, A1 = reaction_picture()
assert round(Q_fis) == 173 and A0 == A1 == 236                                    # no nucleon is lost: only mass
qs = {k: q_value(*v) for k, v in REACTIONS.items()}
assert round(qs["fusion in the Sun, in all: 4 H-1 → He-4"], 1) == 26.7 and round(qs["fusion: H-2 + H-3 → He-4 + n"], 1) == 17.6
assert round(qs["alpha decay: Po-210 → Pb-206 + He-4"], 2) == 5.41 and round(qs["fission: U-235 + n → Xe-140 + Sr-94 + 2n"]) == 185
assert round(qs["fusion: H-2 + H-2 → He-3 + n"], 2) == 3.27 and round(qs["alpha decay: Ra-226 → Rn-222 + He-4"], 2) == 4.87
# the same fission energy from the curve: the pieces' binding energies minus uranium's
assert close(binding_energy("Ba", 141) + binding_energy("Kr", 92) - binding_energy("U", 235), Q_fis, 1e-9)
# the Sun's chain, step by step (the positron's sum uses the electron's mass, 0.000549 u): 0.42, 1.02, 5.49, 12.86 MeV
m_e = 5.485799090441e-4
step1 = (2 * M_H - mass("H", 2) - 2 * m_e) * U_MEV; meet = 2 * m_e * U_MEV
step2 = q_value([("H", 2), ("H", 1)], [("He", 3)]); step3 = q_value([("He", 3), ("He", 3)], [("He", 4), ("H", 1), ("H", 1)])
assert (round(step1, 2), round(meet, 2), round(step2, 2), round(step3, 2)) == (0.42, 1.02, 5.49, 12.86)
assert close(2 * (step1 + meet + step2) + step3, qs["fusion in the Sun, in all: 4 H-1 → He-4"], 1e-9)

In [ ]:
interact(reaction_picture, reaction=Dropdown(options=list(REACTIONS), description="reaction", layout=Layout(width="560px")));

## The Sun's hydrogen

The course's opening question. The Sun's power is 3.828 × 10²⁶ W. By E = mc², that much energy each second is
4.26 × 10⁹ kg of mass each second. Each helium nucleus the Sun makes, from four hydrogen nuclei, releases 26.73 MeV, so
divide the power by that energy to count the helium made each second; four hydrogen atoms' mass for each gives the
hydrogen used: about 600 million tonnes a second. Run backwards: if a star can use a tenth of its mass this way, how
long does it shine? Move the sliders to try other stars (a star twice as heavy as the Sun shines several times as
brightly, and burns out sooner).

In [ ]:
def sun_picture(power_suns=1.0, mass_suns=1.0, usable=0.1):
    P = power_suns * L_SUN
    Q = q_value([("H", 1)] * 4, [("He", 4)]) * MEV                                  # joules for each helium made
    rate = P / Q; hydrogen = rate * 4 * M_H * U_KG; helium = rate * mass("He", 4) * U_KG; light = P / C ** 2
    years = usable * mass_suns * M_SUN / hydrogen / YEAR
    fig, ax = plt.subplots(figsize=(10, 3.6)); fig.subplots_adjust(left=0.17, right=0.96, top=0.78, bottom=0.22)
    ax.barh(["leaves as light", "helium made", "hydrogen used"], [light / 1e9, helium / 1e9, hydrogen / 1e9],
            color=[VIOLET, CHALK, GOLD], height=0.55)
    ax.set_xlabel("million tonnes each second"); ax.grid(False)
    fig.suptitle(f"hydrogen {hydrogen / 1e9:,.0f} · helium {helium / 1e9:,.0f} · light {light / 1e9:.2f} million tonnes a second"
                 f"   ·   lasts {years / 1e9:.1f} billion years", x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own rough model: a tenth of the star's mass is hydrogen it can use"); plt.show()
    return hydrogen, helium, light, years

hyd, hel, light, years = sun_picture()
assert round(light / 1e9, 2) == 4.26 and round(hyd / 1e9) == 598 and round(hel / 1e9) == 594
assert close(hyd - hel, light, 1e-9) and 10e9 < years < 11e9
share = (4 * M_H - mass("He", 4)) / (4 * M_H)
assert round(100 * share, 2) == 0.71 and close(light / share, hyd, 1e-9)           # the same answer another way
fire_years = M_SUN / (M_H * U_KG) * 13.6 * EV / L_SUN / YEAR
assert 2.0e5 < fire_years < 2.3e5                                                  # as a fire, even at 13.6 eV an atom: 200,000 years

In [ ]:
interact(sun_picture, power_suns=FloatSlider(value=1.0, min=0.1, max=20.0, step=0.1, description="power (Suns)"),
         mass_suns=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description="mass (Suns)"),
         usable=FloatSlider(value=0.1, min=0.02, max=0.3, step=0.01, description="usable share"));

## Sources
Every exercise and set-up here is our own. The constants are CODATA 2022 (physics.nist.gov/cuu/Constants). The atomic
masses are from AME2020, the 2020 atomic mass evaluation (Chinese Physics C 45, 030003, 2021; the Atomic Mass Data
Center's file mass_1.mas20). The measured hydrogen lines are NIST's, in air. The Sun's power is the International
Astronomical Union's nominal value (2015, Resolution B3). The alpha is polonium-210's (5.3 MeV); gold's atomic radius
is 144 pm. R₀ = 1.2 fm is a typical measured value. "About 200 MeV" for each fission is the usual total once the pieces
have settled; neutrinos carry off about 2 % of the Sun's energy, so the real Sun uses slightly more hydrogen than the
sum here. Each path is stepped by fourth-order Runge–Kutta.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph22_atoms_nuclei/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph22_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        assert _q["tol"] == _flat(_e["tol"]), _e["id"]
    assert len(_P.EX) == len(EX) and MASS == _P.MASS and MEASURED == _P.MEASURED_NM
    assert (E_CH, H, C, EPS0, M_E, U_KG, U_MEV) == (_P.E_CH, _P.H, _P.C, _P.EPS0, _P.M_E, _P.U_KG, _P.U_MEV)
    assert close(_P.ALPHA["d"], d_gold, 1e-12) and close(_P.SUN["H_rate"], hyd, 1e-4) and close(_P.FISSION["Q"], Q_fis, 1e-12)
    assert close(_P.PUD["max_deg"], worst, 5e-3) and close(_P.BALMER[(3, 2)], nm32, 1e-12)
    print("matches the course's practice set")